# SummitBridge Health Plan - High-Cost Claimant Analysis

**Analysis Period**: January 1, 2024 – December 31, 2024
**Notebook**: 04_exploratory_analysis/02_high_cost_claimants.ipynb
**Purpose**: Identify top 5% members by allowed spend, analyze cost drivers, demographics, and patterns

In [ ]:
# Setup
import sys
sys.path.append('../../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from src.data_loader import load_all
from src.utils import *

print_section("LOADING DATA")
data = load_all()
claims = data['claims']
enrollment = data['enrollment']
member_months = data['member_months']
providers = data['providers']
avoidable_ed = data['avoidable_ed']

print(f"Claims: {len(claims):,} rows")
print(f"Enrollment: {len(enrollment):,} rows")

In [ ]:
# High-Cost Threshold Calculation
print_section("HIGH-COST THRESHOLD (Top 5%)")

member_total = claims.groupby('member_id')['allowed_amount'].sum().reset_index()
member_total.columns = ['member_id', 'total_allowed']

hc_threshold = member_total['total_allowed'].quantile(0.95)
high_cost_members = member_total[member_total['total_allowed'] >= hc_threshold]['member_id'].tolist()
hc_claims = claims[claims['member_id'].isin(high_cost_members)]

total_allowed_all = member_total['total_allowed'].sum()
hc_total_allowed = hc_claims['allowed_amount'].sum()

print(f"High-Cost Threshold (Top 5%): ${hc_threshold:,.2f}")
print(f"High-Cost Members: {len(high_cost_members)}")
print(f"High-Cost Total Allowed: ${hc_total_allowed:,.2f}")
print(f"Total Allowed (All): ${total_allowed_all:,.2f}")
print(f"High-Cost Concentration: {hc_total_allowed/total_allowed_all*100:.1f}% of spend from 5% of members")

In [ ]:
# High-Cost Concentration Visualization
hc_data = pd.DataFrame({
    'Group': ['Top 5% (High-Cost)', 'Remaining 95%'],
    'Allowed': [hc_total_allowed, total_allowed_all - hc_total_allowed],
    'Member_Count': [len(high_cost_members), len(member_total) - len(high_cost_members)]
})

fig = make_subplots(rows=1, cols=2, subplot_titles=('Share of Allowed Spend', 'Share of Members'), specs=[[{'type':'pie'}, {'type':'pie'}]])

In [ ]:
fig.add_trace(go.Pie(labels=hc_data['Group'], values=hc_data['Allowed'], name='Spend', marker_colors=['#d62728', '#2ca02c']), 1, 1)
fig.add_trace(go.Pie(labels=hc_data['Group'], values=hc_data['Member_Count'], name='Members', marker_colors=['#d62728', '#2ca02c']), 1, 2)
fig.update_layout(title_text='High-Cost Member Concentration (Top 5% = 20.5% of Spend)', height=500)
fig.show()

In [ ]:
# High-Cost by Service Category
print_section("HIGH-COST CLAIMS BY SERVICE CATEGORY")

hc_cat = hc_claims.groupby('service_category').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean'),
    pct_of_hc_total=('allowed_amount', lambda x: x.sum()/hc_total_allowed*100)
).sort_values('total_allowed', ascending=False)

print(hc_cat.to_string())

In [ ]:
# Visualization: HC Spend by Service Category
fig = px.bar(
    hc_cat.reset_index(),
    x='service_category',
    y='total_allowed',
    title='High-Cost Member Spend by Service Category',
    labels={'total_allowed': 'Total Allowed ($)', 'service_category': 'Service Category'},
    color='service_category',
    color_discrete_sequence=COLOR_SEQ,
    text_auto='.2s'
)
fig.update_layout(height=400, showlegend=False)
fig.show()

In [ ]:
# High-Cost by Line of Business
print_section("HIGH-COST CLAIMS BY LINE OF BUSINESS")

hc_lob = hc_claims.groupby('lob').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean'),
    unique_members=('member_id', 'nunique')
).sort_values('total_allowed', ascending=False)

print(hc_lob.to_string())

In [ ]:
# High-Cost Member Demographics
print_section("HIGH-COST MEMBER DEMOGRAPHICS")

hc_enrollment = enrollment[enrollment['member_id'].isin(high_cost_members)]

print(f"Average Age: {hc_enrollment['age'].mean():.1f}")
print(f"Age Range: {hc_enrollment['age'].min()} - {hc_enrollment['age'].max()}")
print(f"\nGender Distribution:")
print(hc_enrollment['gender'].value_counts().to_string())
print(f"\nLOB Distribution:")
print(hc_enrollment['lob'].value_counts().to_string())
print(f"\nState Distribution:")
print(hc_enrollment['state'].value_counts().to_string())
print(f"\nAverage Risk Score: {hc_enrollment['risk_score'].mean():.2f}")
print(f"Disenrolled: {hc_enrollment['disenrollment_reason'].notna().sum()} ({hc_enrollment['disenrollment_reason'].notna().sum()/len(hc_enrollment)*100:.1f}%)")

In [ ]:
# High-Cost Member Detail Table
print_section("TOP 20 HIGH-COST MEMBERS")

hc_member_detail = hc_claims.groupby('member_id').agg(
    total_allowed=('allowed_amount', 'sum'),
    claim_count=('claim_id', 'count'),
    ip_claims=('claim_id', lambda x: (hc_claims.loc[x.index, 'service_category'] == 'Inpatient').sum()),
    ed_claims=('claim_id', lambda x: (hc_claims.loc[x.index, 'service_category'] == 'ED').sum()),
    specialty_rx_claims=('claim_id', lambda x: (hc_claims.loc[x.index, 'service_category'] == 'Specialty Rx').sum()),
).sort_values('total_allowed', ascending=False)

# Merge demographics
hc_member_detail = hc_member_detail.merge(
    hc_enrollment[['member_id', 'age', 'gender', 'lob', 'state', 'risk_score']],
    on='member_id', how='left'
)

print(hc_member_detail.head(20).to_string())

In [ ]:
# High-Cost by State
print_section("HIGH-COST BY STATE")

hc_state = hc_claims.groupby('state').agg(
    claim_count=('claim_id', 'count'),
    total_allowed=('allowed_amount', 'sum'),
    avg_allowed=('allowed_amount', 'mean'),
    unique_members=('member_id', 'nunique')
).sort_values('total_allowed', ascending=False)

print(hc_state.to_string())

In [ ]:
# Risk Score Distribution: HC vs Non-HC
print_section("RISK SCORE: HIGH-COST vs NON-HIGH-COST")

non_hc_members = member_total[~member_total['member_id'].isin(high_cost_members)]['member_id'].tolist()
non_hc_enrollment = enrollment[enrollment['member_id'].isin(non_hc_members)]

print(f"High-Cost Avg Risk Score: {hc_enrollment['risk_score'].mean():.2f}")
print(f"Non-High-Cost Avg Risk Score: {non_hc_enrollment['risk_score'].mean():.2f}")
print(f"Ratio: {hc_enrollment['risk_score'].mean() / non_hc_enrollment['risk_score'].mean():.1f}x")